# How Retrieval Works in RAG

> **From a user's question to the most relevant pieces of knowledge.**

In the previous tutorials, we built our understanding of the RAG architecture and explored the indexing pipeline.

We saw how raw documents are transformed into searchable representations:

```text
Documents
    ↓
Ingestion
    ↓
Chunking
    ↓
Embeddings
    ↓
Index
```

But this leaves us with an important question:

> **When a user asks a question, how does the system decide which pieces of information to retrieve?**

That's the job of the **retrieval pipeline**.

In this tutorial, we'll build a simple semantic retriever, inspect its results, and investigate why retrieval is one of the most important—and most frequently underestimated—parts of a RAG system.

## What We'll Learn

By the end of this tutorial, you'll understand:

- What retrieval means in a RAG system
- How a user query becomes a searchable representation
- How query and document embeddings are compared
- What similarity scores represent
- How `top-k` retrieval works
- Why the highest-scoring result isn't always the best result
- How retrieval can fail
- The difference between retrieval precision and recall
- Why retrieval quality directly affects generation quality
- Why we need better retrieval strategies as our corpus grows

We'll start with a deliberately simple retriever and use experiments to understand its behavior before introducing more sophisticated retrieval methods.

# 1. The Retrieval Problem

Imagine that our knowledge base contains 100,000 chunks.

A user asks:

> **"How long do I have to request a refund?"**

We clearly don't want to send all 100,000 chunks to the LLM.

Instead, we need a mechanism that can answer:

> **Which pieces of our knowledge are most relevant to this question?**

That's retrieval.

At its simplest:

```text
100,000 chunks
      ↓
   Retriever
      ↓
  Top 5 chunks
```

Those five chunks can then be supplied to the generation model.

So retrieval acts as a bridge between our external knowledge and the LLM.

---
# 2. Retrieval Is a Separate Problem

It's useful to separate retrieval from generation.

A RAG system contains at least two major tasks:

### Retrieval

> Find useful information.

### Generation

> Use that information to produce an answer.

```text
                    RAG

User Question
      │
      ▼
  RETRIEVAL
      │
      ▼
Relevant Information
      │
      ▼
  GENERATION
      │
      ▼
    Answer
```

This distinction is extremely important.

If the correct information never reaches the LLM, the LLM cannot reliably use it.

```text
Bad Retrieval
     ↓
Wrong Context
     ↓
Potentially Wrong Answer
```

Improving the LLM does not necessarily solve a retrieval problem.

# 3. Our Retrieval Dataset

Now we'll create a slightly larger dataset than the previous notebook.

The goal is to have documents that are similar enough to make retrieval interesting.

In [1]:
documents = [
    {
        "id": "refund_policy",
        "text": "Customers can request a refund within 30 days of purchase."
    },
    {
        "id": "refund_processing",
        "text": "Approved refunds are normally processed within 7 business days."
    },
    {
        "id": "refund_condition",
        "text": "Products must be returned in their original condition to qualify for a refund."
    },
    {
        "id": "shipping_standard",
        "text": "Standard shipping normally takes between 3 and 5 business days."
    },
    {
        "id": "shipping_express",
        "text": "Express shipping normally takes between 1 and 2 business days."
    },
    {
        "id": "shipping_tracking",
        "text": "Customers can track orders using the tracking number provided by email."
    },
    {
        "id": "support_hours",
        "text": "Customer support is available Monday through Friday from 9 AM to 5 PM."
    }
]

For now, think of each entry as a chunk in our knowledge base.

Later we'll work with real documents and much larger corpora.

---
# 4. Generate Document Embeddings

We need to convert each chunk into a vector.

We'll use the same embedding model from the previous notebook.

If you're opening this notebook independently, install the dependency:

In [2]:
# !pip install sentence-transformers

Using `!pip` here is intentional. The course keeps lesson-specific dependencies inside the notebooks so that the same setup can be reproduced in local JupyterLab or Google Colab.

Then load the model:

In [3]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Now generate embeddings for our chunks:

In [4]:
texts = [document["text"] for document in documents]

document_embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True
)

Let's inspect the shape:

In [5]:
document_embeddings.shape

(7, 384)

The result has the conceptual form:

```text
(number_of_chunks, embedding_dimension)
```

We now have:

```text
Document 1 → Vector
Document 2 → Vector
Document 3 → Vector
...
```

The retrieval system will compare a user's query vector against these document vectors.

---
# 5. Turn the User's Question Into a Vector

Our user asks:

> **"How long do I have to request a refund?"**

We'll encode the question using the **same embedding model**.

In [6]:
query = "How long do I have to request a refund?"

query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
)

We now have two things:

```text
Knowledge Base:

Chunk → Embedding
Chunk → Embedding
Chunk → Embedding
...

User:

Query → Embedding
```

The next step is to compare them.

---
# 6. Comparing the Query With Our Knowledge

We need a way to measure how closely the query relates to each chunk.

Because our vectors are normalized, we can calculate cosine similarity using a dot product:

In [7]:
import numpy as np

scores = document_embeddings @ query_embedding

Let's inspect the scores:

In [8]:
for document, score in zip(documents, scores):
    print(f"{score:.4f}  |  {document['id']}")
    print(document["text"])
    print()

0.8186  |  refund_policy
Customers can request a refund within 30 days of purchase.

0.8045  |  refund_processing
Approved refunds are normally processed within 7 business days.

0.7495  |  refund_condition
Products must be returned in their original condition to qualify for a refund.

0.6311  |  shipping_standard
Standard shipping normally takes between 3 and 5 business days.

0.6193  |  shipping_express
Express shipping normally takes between 1 and 2 business days.

0.4982  |  shipping_tracking
Customers can track orders using the tracking number provided by email.

0.5878  |  support_hours
Customer support is available Monday through Friday from 9 AM to 5 PM.



The exact scores will depend on the embedding model.

What matters is the ranking.

A higher score indicates that the embedding model considers the query and chunk more similar in the embedding space.

# 7. Ranking the Results

We don't just want similarity scores.

We want the most relevant chunks first.

In [9]:
ranked_indices = np.argsort(scores)[::-1]

Let's display the ranking:

In [10]:
for rank, index in enumerate(ranked_indices, start=1):
    document = documents[index]

    print(f"Rank {rank}")
    print(f"Score: {scores[index]:.4f}")
    print(f"ID: {document['id']}")
    print(f"Text: {document['text']}")
    print()

Rank 1
Score: 0.8186
ID: refund_policy
Text: Customers can request a refund within 30 days of purchase.

Rank 2
Score: 0.8045
ID: refund_processing
Text: Approved refunds are normally processed within 7 business days.

Rank 3
Score: 0.7495
ID: refund_condition
Text: Products must be returned in their original condition to qualify for a refund.

Rank 4
Score: 0.6311
ID: shipping_standard
Text: Standard shipping normally takes between 3 and 5 business days.

Rank 5
Score: 0.6193
ID: shipping_express
Text: Express shipping normally takes between 1 and 2 business days.

Rank 6
Score: 0.5878
ID: support_hours
Text: Customer support is available Monday through Friday from 9 AM to 5 PM.

Rank 7
Score: 0.4982
ID: shipping_tracking
Text: Customers can track orders using the tracking number provided by email.



---
This is the basic retrieval operation:

```text
Query
  ↓
Query Embedding
  ↓
Compare Against Document Embeddings
  ↓
Similarity Scores
  ↓
Sort
  ↓
Ranked Results
```

# 8. Top-K Retrieval

In a real RAG system, we usually don't return every result.

Instead, we select the top `k`.

In [12]:
top_k = 3

top_results = [
    {
        **documents[index],
        "score": float(scores[index])
    }
    for index in ranked_indices[:top_k]
]

Now inspect them:

In [13]:
for result in top_results:
    print(f"{result['score']:.4f} | {result['id']}")
    print(result["text"])
    print("---")

0.8186 | refund_policy
Customers can request a refund within 30 days of purchase.
---
0.8045 | refund_processing
Approved refunds are normally processed within 7 business days.
---
0.7495 | refund_condition
Products must be returned in their original condition to qualify for a refund.
---


---
The value of `k` is an important retrieval parameter.

For example:

```text
k = 1
```

means we retrieve one result.

```text
k = 5
```

means we retrieve five.

```text
k = 20
```

means we retrieve twenty.

But bigger isn't automatically better.

---
# 9. Why More Retrieved Results Can Be Worse

It might seem logical that retrieving more information should improve the answer.

But consider:

```text
Top 3

✓ Refund deadline
✓ Refund eligibility
✓ Refund processing time
```

while:

```text
Top 20

✓ Refund information
✓ More refund information
✗ Shipping
✗ Customer support
✗ Order tracking
✗ Unrelated policies
...
```

We have now introduced noise.

The LLM has to process more information and distinguish relevant evidence from irrelevant information.

So there is a trade-off:

```text
Too few results
      ↓
Missing evidence

Too many results
      ↓
More noise
```

Finding an appropriate retrieval strategy is therefore an engineering problem, not simply a matter of choosing the largest `k`.

# 10. When Retrieval Gets It Wrong

Now let's ask a more difficult question:

> **"What condition must a product meet to qualify for a refund?"**

In [14]:
query = "What condition must a product meet to qualify for a refund?"

query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
)

scores = document_embeddings @ query_embedding

ranked_indices = np.argsort(scores)[::-1]

for rank, index in enumerate(ranked_indices[:5], start=1):
    print(f"Rank {rank} | Score: {scores[index]:.4f}")
    print(documents[index]["text"])
    print()

Rank 1 | Score: 0.8721
Products must be returned in their original condition to qualify for a refund.

Rank 2 | Score: 0.7650
Customers can request a refund within 30 days of purchase.

Rank 3 | Score: 0.7115
Approved refunds are normally processed within 7 business days.

Rank 4 | Score: 0.5969
Customer support is available Monday through Friday from 9 AM to 5 PM.

Rank 5 | Score: 0.5613
Standard shipping normally takes between 3 and 5 business days.



---
Inspect the results carefully.

You may find that multiple refund-related chunks rank highly.

That's reasonable.

But semantic similarity doesn't mean the chunk necessarily contains the **specific evidence** required to answer the question.

> **A chunk can be semantically related to a question without actually answering it.**

# 11. Semantic Similarity Is Not the Same as Relevance

Consider the question:

> "What condition must the product be in?"

These two chunks are both about refunds:

```text
Chunk A:
Customers can request a refund within 30 days.

Chunk B:
Products must be returned in their original condition.
```

Both are semantically related to the topic of refunds.

But only one directly answers the question.

This gives us an important distinction:

```text
Topic similarity
       ≠
Answer relevance
```

Embedding-based retrieval gives us a powerful way to find candidates.

But it isn't perfect.

This is one reason production RAG systems often add another retrieval stage, such as **reranking**.

---
# 12. Measuring Retrieval: Recall

We need a way to measure whether our retriever is actually finding the information we need.

One important concept is **retrieval recall**.

Suppose we know that the correct answer is contained in:

```text
Chunk B
```

If our retriever returns:

```text
Chunk A
Chunk C
Chunk D
```

but not Chunk B, then retrieval has failed to retrieve the relevant evidence.

A useful question is:

> **Did the retriever retrieve the relevant information at all?**

At a high level:

```text
Relevant information that exists
             ↓
Relevant information retrieved
```

If the relevant chunk never enters our candidate set, later stages cannot recover it.

This is why high retrieval recall is often an important goal for the initial retrieval stage.

---
# 13. Measuring Retrieval: Precision

Recall asks:

> Did we retrieve the relevant information?

Precision asks a different question:

> **How much of what we retrieved was actually relevant?**

Imagine we retrieve five chunks:

```text
Chunk 1 ✓
Chunk 2 ✓
Chunk 3 ✗
Chunk 4 ✗
Chunk 5 ✗
```

Only two are relevant.

The retriever found useful information, but it also returned considerable noise.

This gives us the basic tension:

```text
Recall
  ↑
Retrieve more candidates

Precision
  ↑
Retrieve fewer, more relevant candidates
```

In practice, RAG retrieval often involves balancing these goals across multiple stages.

# 14. Retrieval and Generation Are Connected

Suppose the correct information exists in our knowledge base:

```text
Knowledge Base
      │
      └── Correct answer
```

But retrieval fails:

```text
Knowledge Base
      │
      └── Correct answer
             ✗
          Not retrieved
```

The LLM receives:

```text
Wrong context
```

and we ask:

> "Why did the LLM hallucinate?"

But the real problem may have occurred earlier.

```text
Correct information
       ↓
Poor retrieval
       ↓
Missing evidence
       ↓
LLM
       ↓
Unsupported answer
```

This is why we need to evaluate RAG **component by component**.

# 15. The Retrieval-Generation Boundary

At this point, it's useful to establish a boundary:

```text
              RETRIEVAL
                  │
                  ▼
            Retrieved Context
                  │
══════════════════╪══════════════════
                  │
                  ▼
              GENERATION
                  │
                  ▼
                Answer
```

The retrieval system is responsible for finding useful evidence.

The generation system is responsible for using that evidence.

This separation makes debugging much easier.

If the correct chunk wasn't retrieved:

> Investigate retrieval.

If the correct chunk was retrieved but the LLM ignored or misinterpreted it:

> Investigate generation/context handling.

# 16. Our Retriever So Far

We've implemented:

```text
Query
  ↓
Embedding
  ↓
Cosine Similarity
  ↓
Ranking
  ↓
Top-K
```

In code, the essential operation is surprisingly small:

In [15]:
query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
)

scores = document_embeddings @ query_embedding

ranked_indices = np.argsort(scores)[::-1]

top_results = ranked_indices[:top_k]

But the simplicity of this code hides several important engineering questions:

- Is the embedding model good enough?
- Is our chunking good enough?
- Is semantic search enough?
- How should we choose `k`?
- How do we evaluate retrieval?
- What happens when exact keywords matter?
- What happens when the query is ambiguous?
- What happens when multiple documents contain conflicting information?

Those questions lead us toward more advanced retrieval architectures.

---
# 17. The Limitation of Dense Retrieval

Our current retriever is based on embeddings.

This is commonly called **dense retrieval**.

It is excellent at capturing semantic relationships.

But semantic similarity isn't always what we need.

Consider a query containing an exact identifier:

```text
"Policy ID: REF-2026-00491"
```

A user might expect the system to find the exact identifier.

Keyword-based search can be extremely useful for these cases.

This gives us two complementary approaches:

```text
Dense Retrieval
    ↓
Semantic similarity

Keyword Retrieval
    ↓
Lexical matching
```

Rather than choosing one blindly, production systems can combine them.

That will lead us to **hybrid retrieval**.

---
# 18. From Simple Retrieval to Production Retrieval

Our baseline:

```text
Query
  ↓
Dense Embedding
  ↓
Vector Similarity
  ↓
Top-K
```

A more advanced architecture can look like:

```text
                     Query
                       │
              ┌────────┴────────┐
              ↓                 ↓
        Dense Retrieval    Keyword Retrieval
              ↓                 ↓
         Candidates          Candidates
              └────────┬────────┘
                       ↓
                      RRF
                       ↓
                  Reranking
                       ↓
                Final Context
```

This architecture gives us several opportunities to improve retrieval quality.

But we shouldn't add all of these components simply because they exist.

We'll introduce them when we understand the problem they solve.

# 19. Inspect Retrieval Before Blaming the LLM

When a RAG system produces a bad answer, don't immediately change the prompt or model.

First ask:

> **What did the retriever actually return?**

For every query, inspect:

```text
Query
 ↓
Retrieved Chunk 1
 ↓
Retrieved Chunk 2
 ↓
Retrieved Chunk 3
```

Then ask:

1. Is the relevant information present?
2. Is it ranked highly enough?
3. Are irrelevant chunks dominating?
4. Is the information split across chunks?
5. Is the source trustworthy?
6. Is there conflicting information?

This simple debugging habit can save a huge amount of time.

A generated answer is downstream of retrieval.

So inspect the evidence first.

# 20. The Retrieval Pipeline We've Built

We can now summarize our implementation:

```text
                 USER QUERY
                     │
                     ▼
               Query Embedding
                     │
                     ▼
             Similarity Search
                     │
                     ▼
                Score Chunks
                     │
                     ▼
                  Ranking
                     │
                     ▼
                   Top-K
                     │
                     ▼
             Retrieved Context
```

This is the simplest form of semantic retrieval.

The next stages of the course will make this pipeline much more robust.

# Key Takeaways

1. **Retrieval finds information; generation produces the answer.**
2. A user query can be converted into an embedding and compared with document embeddings.
3. Similarity scores allow candidate chunks to be ranked.
4. `top-k` determines how many candidates are retrieved.
5. Retrieving more information is not automatically better because additional results can introduce noise.
6. **Semantic similarity does not guarantee answer relevance.**
7. Retrieval recall asks whether relevant information was retrieved.
8. Retrieval precision asks how much of the retrieved information was actually relevant.
9. If the correct evidence is not retrieved, the generation stage cannot reliably use it.
10. Dense retrieval is powerful but has limitations, especially for exact lexical matches.
11. A production RAG system may combine multiple retrieval strategies.
12. Always inspect retrieved evidence before blaming the LLM for a poor answer.

The central mental model is:

```text
Query
  ↓
Retrieve Evidence
  ↓
Evaluate Evidence
  ↓
Build Context
  ↓
Generate Answer
```

# What's Next?

We've now built a basic semantic retriever and seen both its strengths and weaknesses.

But we've been doing something unrealistic:

```text
Python
  ↓
Store all vectors in memory
  ↓
Calculate similarities ourselves
```

That works for a handful of chunks.

It doesn't work well when our knowledge base contains millions of vectors.

We therefore need infrastructure designed specifically for storing and searching vector representations.

In the next tutorial, we'll explore:

> **Vector Databases: What They Actually Do and Why We Need Them**

We'll introduce **Qdrant**, connect it to our RAG pipeline, and see how our simple in-memory retriever becomes a proper searchable vector store.